# Topic: SQL Pivot Table Pattern

## Definition (30-second explanation)
* The Pivot operation transforms rows into columns, reshaping data from a long/tall format to a wide format.
* Because most databases (like MySQL and PostgreSQL) lack a native `PIVOT` keyword, the standard approach is conditional aggregation: using `CASE WHEN` inside aggregation functions like `SUM()`, `COUNT()`, or `MAX()`.

## Why Interviewers Ask This
* Pivoting is a fundamental data preparation step for dashboards, BI tools (Tableau, PowerBI), and spreadsheet exports.
* It tests your understanding of the `GROUP BY` clause and how aggregation functions evaluate row-level logic.

## Core Concepts
* **Conditional Aggregation:** The core engine of a pivot. E.g., `SUM(CASE WHEN category = 'A' THEN value ELSE 0 END)`.
* **Row Identifier:** The column(s) placed in the `GROUP BY` clause become the unique row identifiers in the final wide table.
* **Native vs Universal:** SQL Server and Oracle have a native `PIVOT()` operator, but the `CASE WHEN` approach is universal across almost all relational databases.

## When to Use
* Converting monthly or quarterly metric rows into side-by-side columns for management reports.
* Turning category-value (EAV model) pairs into dedicated columns for machine learning features.
* Reshaping survey response data (one row per answer) into one row per respondent.
* Creating crosstab/matrix reports.

## Advantages
* The `CASE WHEN` pivot is universally supported and highly readable.
* Allows for complex transformations during the pivot (e.g., calculating YoY growth between the newly created columns in the same query).

## Limitations
* **Hardcoded Columns:** Standard SQL pivots require you to know all possible column values at query-writing time. 
* **Dynamic Pivoting:** If the column values change dynamically (e.g., new product names added daily), standard SQL cannot adapt automatically; you must use dynamic SQL (constructing query strings programmatically).

## Common Comparisons
* **Pivot vs. Unpivot:** Pivot turns rows into columns (long to wide). Unpivot (See 'Unpivot Pattern') turns columns into rows (wide to long).
* **`SUM()` vs `MAX()` in Pivots:** Use `SUM()` when aggregating numerical metrics (like revenue). Use `MAX()` when pivoting categorical/string data (like survey answers) to pull the single text value into the column.

## Common Interview Traps
* **Missing the Row Identifier:** Forgetting to include the identifier (e.g., `department`) in the `GROUP BY` clause collapses the entire dataset into a single row.
* **Using `ELSE NULL` instead of `ELSE 0`:** When pivoting numerical sums, omitting the `ELSE 0` (or explicitly using `ELSE NULL`) will return NULLs for missing categories. This can break downstream mathematical operations.
* **Division by Zero:** When calculating growth between pivot columns, an empty period will evaluate to 0, causing a division by zero error. Always wrap the denominator in `NULLIF(denominator, 0)`.

## Python / SQL Syntax
```sql
    -- Standard Universal Pivot (MySQL, Postgres, BigQuery)
    SELECT 
        department,
        SUM(CASE WHEN quarter = 'Q1' THEN revenue ELSE 0 END) AS Q1_revenue,
        SUM(CASE WHEN quarter = 'Q2' THEN revenue ELSE 0 END) AS Q2_revenue,
        SUM(CASE WHEN quarter = 'Q3' THEN revenue ELSE 0 END) AS Q3_revenue,
        SUM(CASE WHEN quarter = 'Q4' THEN revenue ELSE 0 END) AS Q4_revenue,
        SUM(revenue) AS total_annual_revenue
    FROM sales
    GROUP BY department;
```

## 45-Second Interview Answer
"To pivot data from a long to a wide format in standard SQL, I use conditional aggregation. I group by the desired row identifier, and for each new column, I wrap a `CASE WHEN` statement inside an aggregate function like `SUM` or `MAX`. This evaluates each row, placing the value into the correct column and defaulting to 0 or NULL for the rest. While SQL Server has a native PIVOT operator, this `CASE WHEN` method is universal. The main limitation is that you must hardcode the columns; for fully dynamic pivoting, I would need to write a stored procedure using dynamic SQL or handle it in the BI/Pandas layer."

## Example Questions:

### Q1: Pivot monthly sales data (Jan-Dec) into columns. Also calculate the month with the highest sales using MAX() and CASE WHEN.

* **Ideal Interview Answer:** I'd group by the relevant dimension (e.g., year or store) and write 12 `SUM(CASE WHEN month = 'Jan'...)` statements. 
```sql
    SELECT store_id,
           SUM(CASE WHEN month = 'Jan' THEN sales ELSE 0 END) AS Jan_sales,
           SUM(CASE WHEN month = 'Feb' THEN sales ELSE 0 END) AS Feb_sales,
           -- ... (Mar through Nov)
           SUM(CASE WHEN month = 'Dec' THEN sales ELSE 0 END) AS Dec_sales,
           MAX(sales) AS highest_monthly_sales
    FROM monthly_sales
    GROUP BY store_id;
```
* **Common Mistakes:** Using `ELSE NULL` for numerical sales data instead of `ELSE 0`.
* **Likely Interviewer Follow-up:** The prompt asked for the *month* with the highest sales, but `MAX(sales)` just returns the highest *value*. How would you get the actual month name? (Answer: I would need a CTE with a window function like `RANK() OVER(PARTITION BY store_id ORDER BY sales DESC)` to flag the top month, and then pivot that alongside the data, or use a complex `GREATEST` comparison across the pivoted columns).

### Q2: Transform a survey table (user_id, question_id, answer) into one row per user with columns for each question.

* **Ideal Interview Answer:** Because survey answers are typically text/strings, I cannot use `SUM()`. Instead, I will use `MAX()` as the aggregation function to pull the string value into the pivoted column.
```sql
    SELECT user_id,
           MAX(CASE WHEN question_id = 1 THEN answer END) AS question_1_answer,
           MAX(CASE WHEN question_id = 2 THEN answer END) AS question_2_answer,
           MAX(CASE WHEN question_id = 3 THEN answer END) AS question_3_answer
    FROM survey_responses
    GROUP BY user_id;
```
* **Common Mistakes:** Trying to use `SUM()` on text columns, which will result in an error or `0`.
* **Likely Interviewer Follow-up:** Why does `MAX()` work for strings here? (Answer: When grouped by `user_id`, there is only one answer per `question_id`. The `CASE WHEN` makes all other rows NULL for that specific column. `MAX()` ignores NULLs and simply returns the single non-NULL string value).

### Q3: Create a report showing, for each product category, how many products are in price ranges: under 100, 100-500, 500-1000, over 1000.

* **Ideal Interview Answer:** I will group by product category and use conditional counting. I prefer using `SUM` with 1 and 0 for this type of bucketing pivot.
```sql
    SELECT category,
           SUM(CASE WHEN price < 100 THEN 1 ELSE 0 END) AS under_100,
           SUM(CASE WHEN price >= 100 AND price < 500 THEN 1 ELSE 0 END) AS range_100_to_500,
           SUM(CASE WHEN price >= 500 AND price <= 1000 THEN 1 ELSE 0 END) AS range_500_to_1000,
           SUM(CASE WHEN price > 1000 THEN 1 ELSE 0 END) AS over_1000
    FROM products
    GROUP BY category;
```
* **Common Mistakes:** Using `COUNT(CASE WHEN price < 100 THEN 1 ELSE 0 END)`. `COUNT()` counts non-nulls, so it would count the `0`s as well. To use `COUNT`, you must remove the `ELSE 0` so it defaults to NULL.
* **Likely Interviewer Follow-up:** How would you ensure all categories appear in the report, even if they have zero products in these ranges? (Answer: Ensure the `products` table is `LEFT JOIN`ed from a master `categories` table before doing the pivot aggregation).

### Q4: Pivot attendance data to show for each employee: days_present, days_absent, days_on_leave as separate columns.

* **Ideal Interview Answer:** Similar to the previous question, this is a conditional counting pivot. I will group by employee and sum the occurrences of each status.
```sql
    SELECT employee_id,
           SUM(CASE WHEN status = 'Present' THEN 1 ELSE 0 END) AS days_present,
           SUM(CASE WHEN status = 'Absent' THEN 1 ELSE 0 END) AS days_absent,
           SUM(CASE WHEN status = 'Leave' THEN 1 ELSE 0 END) AS days_on_leave
    FROM attendance
    GROUP BY employee_id;
```
* **Common Mistakes:** Forgetting the `GROUP BY employee_id`, resulting in a single row showing the total days present/absent for the entire company.
* **Likely Interviewer Follow-up:** How would you add a column for the employee's attendance percentage? (Answer: I would calculate `SUM(CASE WHEN status = 'Present' THEN 1 ELSE 0 END) / COUNT(*) * 100.0` within the same `SELECT` statement).

## Practice Questions:

### Q1:
**Scenario:**
You are interviewing for a Data Scientist role. The interviewer says: "Our application logs user settings in a very tall, narrow table using an Entity-Attribute-Value (EAV) model. We need to feed this data into a scikit-learn model, so we need it flattened out: one row per user, with their specific settings as columns."

**Your Task:**
Write a MySQL query to transform this table so that the output has exactly three columns: user_id, theme, and language. If a user does not have a language or theme set, return NULL for that column.

**Mock Schema**
```sql
CREATE TABLE user_settings (
    user_id INT,
    setting_name VARCHAR(50),
    setting_value VARCHAR(50)
);

INSERT INTO user_settings (user_id, setting_name, setting_value) VALUES
(101, 'theme', 'dark'),
(101, 'language', 'english'),
(102, 'theme', 'light'),
(102, 'notifications', 'enabled');
```


**Answer:** Because the values we are pivoting are text strings, we cannot use `SUM()`. Instead, I will use `MAX()` as the aggregate function. By omitting the `ELSE` clause in the `CASE WHEN`, unmatched rows default to `NULL`. Since `MAX()` ignores `NULL`s, it will perfectly extract the single string value for that user.
```sql
    SELECT user_id,
           MAX(CASE WHEN setting_name = 'theme' THEN setting_value END) AS theme,
           MAX(CASE WHEN setting_name = 'language' THEN setting_value END) AS language
    FROM user_settings
    GROUP BY user_id;
```
* **Common Mistakes:** Trying to use `SUM()`, which will throw an execution error or return 0 on text data depending on the SQL dialect.
* **Likely Interviewer Follow-up:** How does `MAX()` work on text strings if there happen to be two different rows for the exact same setting (e.g., user 101 has two 'theme' rows)? (Answer: `MAX()` will sort the strings alphabetically and return the one closest to 'Z'. In a proper EAV model, there should be a unique constraint preventing duplicate keys per user, but if duplicates exist, you might need a different strategy like `GROUP_CONCAT()` or `STRING_AGG()` depending on the business need).

### Q2:
**Scenario:**
You are interviewing for a Data Scientist role. The interviewer says: "Our CFO wants a region-by-region revenue report. They want to see Q1 and Q2 revenue as side-by-side columns, but they also want a third column showing the Quarter-over-Quarter (QoQ) percentage growth from Q1 to Q2."

**Your Task:**
Write a MySQL query to return four columns: region_name, Q1_revenue, Q2_revenue, and QoQ_growth_pct.

**Mock Schema**
```sql
CREATE TABLE regional_sales (
    region_name VARCHAR(50),
    quarter VARCHAR(2),
    revenue DECIMAL(10,2)
);

INSERT INTO regional_sales (region_name, quarter, revenue) VALUES
('North America', 'Q1', 100000.00),
('North America', 'Q2', 120000.00),
('Europe', 'Q1', 0.00),  -- Just launched, no Q1 sales
('Europe', 'Q2', 50000.00),
('Asia', 'Q1', 80000.00),
('Asia', 'Q2', 75000.00);
```

**Answer:** I would use a CTE to first perform the conditional aggregation and pivot the Q1 and Q2 revenues into their own columns. In the outer query, I would calculate the growth percentage using those new columns, which makes the code much more readable. To prevent a division-by-zero error if a region had zero revenue in Q1, I will wrap the denominator in a `NULLIF()`.
```sql
    WITH pivot_table AS (
        SELECT region_name,
               SUM(CASE WHEN quarter = 'Q1' THEN revenue ELSE 0 END) AS Q1_revenue,
               SUM(CASE WHEN quarter = 'Q2' THEN revenue ELSE 0 END) AS Q2_revenue
        FROM regional_sales
        GROUP BY region_name
    )
    SELECT region_name,
           Q1_revenue,
           Q2_revenue,
           100.0 * (Q2_revenue - Q1_revenue) / NULLIF(Q1_revenue, 0) AS QoQ_growth_pct
    FROM pivot_table;
```
* **Common Mistakes:** Trying to write the math entirely in one `SELECT` statement without a CTE, resulting in a massive, unreadable block of repeated `SUM(CASE WHEN...)` logic. Forgetting to handle the division by zero, which will crash the query in production if a new market launches with 0 prior sales.
* **Likely Interviewer Follow-up:** If you wrap the whole math equation in `COALESCE(..., 0)`, is that mathematically correct for a business? (Answer: Usually no. A 0% growth implies revenue stayed exactly the same. If Q1 was 0, growth is mathematically undefined. It's usually better to let the database return NULL so the BI tool can render it as 'N/A' or 'New Market' rather than flat growth).

### Q3:
**Scenario:**
You are interviewing for a Data Analyst role. The interviewer says: "Our marketing team wants to compare the performance of our 'Email' and 'Social' campaigns. They need a single row for each campaign containing four metrics: Q1 clicks, Q1 conversions, Q2 clicks, and Q2 conversions."

**Your Task:**
Write a MySQL query to return exactly two rows (one for Email, one for Social) with five columns: campaign_type, Q1_clicks, Q1_conversions, Q2_clicks, and Q2_conversions.

**Mock Schema**
```sql
CREATE TABLE campaign_performance (
    campaign_type VARCHAR(50),
    quarter VARCHAR(2),
    clicks INT,
    conversions INT
);

INSERT INTO campaign_performance (campaign_type, quarter, clicks, conversions) VALUES
('Email', 'Q1', 5000, 250),
('Email', 'Q2', 6000, 300),
('Social', 'Q1', 12000, 150),
('Social', 'Q2', 15000, 200);
```


**Answer:** To pivot multiple metrics, I apply the standard conditional aggregation pattern, but I repeat it for each metric column. I will group by `campaign_type` and create four separate `SUM(CASE WHEN...)` statements, changing the target column in the `THEN` clause to match the metric I am pivoting.
```sql
    SELECT campaign_type,
           SUM(CASE WHEN quarter = 'Q1' THEN clicks ELSE 0 END) AS Q1_clicks,
           SUM(CASE WHEN quarter = 'Q1' THEN conversions ELSE 0 END) AS Q1_conversions,
           SUM(CASE WHEN quarter = 'Q2' THEN clicks ELSE 0 END) AS Q2_clicks,
           SUM(CASE WHEN quarter = 'Q2' THEN conversions ELSE 0 END) AS Q2_conversions
    FROM campaign_performance
    GROUP BY campaign_type;
```
* **Common Mistakes:** Copy-paste errors on column aliases (e.g., naming two columns `Q1_conversions`), which can crash downstream reporting tools. Mixing up the metric being aggregated in the `THEN` clause.
* **Likely Interviewer Follow-up:** If we had 10 different metrics (impressions, clicks, conversions, spend, etc.) across 4 quarters, this query would be 40 lines of `CASE WHEN`. Is there a more efficient way to write this in standard SQL? (Answer: Standard SQL requires explicit declaration, so you'd still need 40 lines. To automate it, you'd have to use a programming language like Python to generate the SQL string dynamically).